# 02 - Multiple Linear Regression End-to-End

Executed notebook for multiple linear regression. It includes EDA, categorical encoding, scikit-learn pipeline, metrics, coefficient interpretation, visual diagnostics, statsmodels inference, and cross-validation.

## 1. Problem statement

Predict `sales_k_units` from TV spend, radio spend, social spend, price index, competitor spend, and season.

For multiple regression, coefficient interpretation must always include: **holding other variables constant**.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.formula.api as smf
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
DATA_DIR = Path.cwd().parent / 'data' if (Path.cwd().parent / 'data').exists() else Path.cwd() / 'data'
print('Libraries imported.')
print('Dataset path resolved.')

Libraries imported.
Dataset path resolved.


## 2. Load and inspect data

In [2]:
df = pd.read_csv(DATA_DIR / 'multiple_regression_marketing_sales.csv')
df.head()

  campaign_id  tv_spend_k  radio_spend_k  social_spend_k  price_index  \
0        C001      146.00          64.71           18.72       116.35   
1        C002       22.08          73.15           36.77       118.69   
2        C003      142.26          65.27           28.03        93.91   
3        C004       54.75          36.33           13.81       116.59   
4        C005       64.26          44.41            9.60       108.44   

   competitor_spend_k season  sales_k_units  
0              164.96     Q2          67.05  
1               71.61     Q3          23.68  
2              138.89     Q1          85.06  
3               27.86     Q1           9.51  
4               77.28     Q4          24.60  

In [3]:
pd.DataFrame({'column': df.columns, 'dtype': [str(dtype) for dtype in df.dtypes], 'missing_count': df.isna().sum().values, 'unique_values': df.nunique().values})

                column    dtype  missing_count  unique_values
0          campaign_id   object              0            120
1           tv_spend_k  float64              0            120
2        radio_spend_k  float64              0            120
3       social_spend_k  float64              0            118
4          price_index  float64              0            120
5   competitor_spend_k  float64              0            120
6               season   object              0              4
7        sales_k_units  float64              0            116

## 3. Summary statistics

In [4]:
df[['tv_spend_k','radio_spend_k','social_spend_k','price_index','competitor_spend_k','sales_k_units']].describe().round(2).T

                    count    mean    std    min    25%     50%     75%     max
tv_spend_k          120.0  153.62  87.75  20.95  74.09  145.76  230.62  316.28
radio_spend_k       120.0   46.09  23.30   5.17  25.63   48.20   66.94   78.80
social_spend_k      120.0   29.64  17.33   1.42  13.61   30.03   44.65   59.96
price_index         120.0  104.38  11.57  85.58  95.15  106.03  114.92  124.95
competitor_spend_k  120.0   91.40  50.41  13.27  44.27   87.39  137.86  178.88
sales_k_units       120.0   74.78  45.88   5.00  35.14   68.41  111.29  173.16

## 4. Visual EDA

In [5]:
season_sales = df.groupby('season')['sales_k_units'].mean().sort_index().round(2)
plt.figure(figsize=(7,4))
plt.bar(season_sales.index, season_sales.values)
plt.title('Average sales by season')
plt.xlabel('Season')
plt.ylabel('Average sales, thousand units')
plt.grid(axis='y', alpha=0.3)
plt.show()
print('Rendered average sales by season bar chart.')
season_sales

season
Q1    47.11
Q2    88.27
Q3    77.94
Q4    89.12
Name: sales_k_units, dtype: float64

Rendered average sales by season bar chart.


In [6]:
numeric_features = ['tv_spend_k', 'radio_spend_k', 'social_spend_k', 'price_index', 'competitor_spend_k']
target = 'sales_k_units'
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for idx, feature in enumerate(numeric_features):
    axes[idx].scatter(df[feature], df[target], alpha=0.7)
    axes[idx].set_title(f'{feature} vs sales')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel(target)
    axes[idx].grid(True, alpha=0.3)
axes[-1].axis('off')
plt.tight_layout()
plt.show()
print('Rendered scatter plots for numeric predictors vs sales.')

Rendered scatter plots for numeric predictors vs sales.


In [7]:
corr = df[numeric_features + [target]].corr().round(2)
fig, ax = plt.subplots(figsize=(8,6))
image = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
ax.set_title('Correlation heatmap')
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()
print('Rendered correlation heatmap.')
corr

                    tv_spend_k  radio_spend_k  social_spend_k  price_index  competitor_spend_k  sales_k_units
tv_spend_k                1.00          -0.08            0.00         0.01               -0.03           0.84
radio_spend_k            -0.08           1.00           -0.06         0.00               -0.01           0.33
social_spend_k            0.00          -0.06            1.00         0.00                0.08           0.16
price_index               0.01           0.00            0.00         1.00               -0.02          -0.14
competitor_spend_k       -0.03          -0.01            0.08        -0.02                1.00          -0.13
sales_k_units             0.84           0.33            0.16        -0.14               -0.13           1.00

Rendered correlation heatmap.


## 5. Preprocessing and model pipeline

`season` is one-hot encoded; numeric variables pass through unchanged.

In [8]:
categorical_features = ['season']
X = df[numeric_features + categorical_features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=43)
preprocess = ColumnTransformer([('categorical', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features), ('numeric', 'passthrough', numeric_features)])
pipeline = Pipeline([('preprocess', preprocess), ('model', LinearRegression())])
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
print('Training rows:', len(X_train))
print('Testing rows :', len(X_test))
print('Pipeline fitted successfully.')

Training rows: 90
Testing rows : 30
Pipeline fitted successfully.


## 6. Model metrics

In [9]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
pd.DataFrame({'metric': ['MAE', 'RMSE', 'R2'], 'value': [mae, rmse, r2]})

  metric     value
0    MAE  6.803410
1   RMSE  9.007675
2     R2  0.959718

In [10]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, pred, alpha=0.8)
line_min = min(y_test.min(), pred.min())
line_max = max(y_test.max(), pred.max())
plt.plot([line_min, line_max], [line_min, line_max], linewidth=2)
plt.title('Actual vs predicted sales')
plt.xlabel('Actual sales')
plt.ylabel('Predicted sales')
plt.grid(True, alpha=0.3)
plt.show()
print('Rendered actual vs predicted plot.')

Rendered actual vs predicted plot.


## 7. Coefficient interpretation

Interpret each coefficient as the expected change in sales **holding other variables constant**.

In [11]:
feature_names = pipeline.named_steps['preprocess'].get_feature_names_out()
coefficients = pipeline.named_steps['model'].coef_
coef_table = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients}).sort_values('coefficient', ascending=False)
coef_table

                       feature  coefficient
2       categorical__season_Q4    16.074842
1       categorical__season_Q3     9.514449
0       categorical__season_Q2     9.473192
4       numeric__radio_spend_k     0.761870
3          numeric__tv_spend_k     0.456565
5      numeric__social_spend_k     0.376357
7 numeric__competitor_spend_k    -0.141192
6        numeric__price_index    -0.744473

In [12]:
coef_plot = coef_table.sort_values('coefficient')
plt.figure(figsize=(9,5))
plt.barh(coef_plot['feature'], coef_plot['coefficient'])
plt.axvline(0, linewidth=1)
plt.title('Multiple regression coefficients')
plt.xlabel('Coefficient value')
plt.ylabel('Feature')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()
print('Rendered coefficient bar chart.')

Rendered coefficient bar chart.


## 8. Error analysis

In [13]:
error_table = X_test.copy()
error_table['actual_sales'] = y_test.values
error_table['predicted_sales'] = pred
error_table['residual'] = error_table['actual_sales'] - error_table['predicted_sales']
error_table['absolute_error'] = error_table['residual'].abs()
error_table.sort_values('absolute_error', ascending=False).round(2).head()

     tv_spend_k  radio_spend_k  social_spend_k  price_index  competitor_spend_k season  actual_sales  predicted_sales  residual  absolute_error
96       133.13          78.25           26.03        124.45               36.01     Q1         59.97            76.37    -16.40           16.40
83        59.38          48.06           25.81        124.91               87.40     Q2         27.18            42.68    -15.50           15.50
53        34.93          18.42           33.53        111.51              167.19     Q1          5.00            19.51    -14.51           14.51
94       145.72          72.28           59.74        116.44              170.86     Q2         96.30           109.93    -13.63           13.63
77       200.15          21.55           29.19        118.31               64.42     Q2         56.22            69.38    -13.16           13.16

In [14]:
residuals = y_test - pred
plt.figure(figsize=(7,5))
plt.scatter(pred, residuals, alpha=0.8)
plt.axhline(0, linewidth=1)
plt.title('Residuals vs predicted sales')
plt.xlabel('Predicted sales')
plt.ylabel('Residual')
plt.grid(True, alpha=0.3)
plt.show()
print('Rendered residual plot.')

Rendered residual plot.


## 9. statsmodels OLS inference

In [15]:
ols = smf.ols('sales_k_units ~ tv_spend_k + radio_spend_k + social_spend_k + price_index + competitor_spend_k + C(season)', data=df).fit()
pd.DataFrame({'coefficient': ols.params, 'p_value': ols.pvalues, 'lower_95': ols.conf_int()[0], 'upper_95': ols.conf_int()[1]}).sort_values('p_value')

                       coefficient       p_value   lower_95   upper_95
tv_spend_k                0.450606  1.983393e-73   0.435466   0.465746
radio_spend_k             0.803542  7.038510e-41   0.728193   0.878891
price_index              -0.749053  2.496985e-16  -0.900427  -0.597678
competitor_spend_k       -0.152480  6.154205e-14  -0.192229  -0.112731
social_spend_k            0.402834  4.656125e-12   0.297521   0.508147
C(season)[T.Q4]          16.733227  4.836192e-10  12.002732  21.463721
C(season)[T.Q2]          12.046460  1.357397e-06   7.322618  16.770302
C(season)[T.Q3]          11.770254  3.909291e-06   6.866316  16.674191
Intercept                39.157222  3.831908e-05  20.987582  57.326863

## 10. Cross-validation

In [16]:
cv = KFold(n_splits=5, shuffle=True, random_state=43)
scores = cross_validate(pipeline, X, y, cv=cv, scoring={'r2': 'r2', 'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'})
cv_results = pd.DataFrame({'fold': range(1, 6), 'r2': scores['test_r2'], 'mae': -scores['test_mae'], 'rmse': -scores['test_rmse']})
cv_results.round(4)

   fold      r2     mae     rmse
0     1  0.9749  5.8933   7.1169
1     2  0.9590  6.6181   8.5653
2     3  0.9516  7.6572   9.8628
3     4  0.9448  8.0391  10.6047
4     5  0.9595  7.2393  10.7706

## Final interpretation

The model explains a high proportion of variation in the synthetic sales target. TV, radio, and social spend have positive associations with sales. Price index and competitor spend have negative associations. Season effects are interpreted relative to the dropped baseline season.

## Student exercises

1. Remove `season` and compare R².
2. Remove `competitor_spend_k` and compare coefficients.
3. Add a residual histogram.
4. Change test size to 30%.
5. Write a business recommendation using the coefficient table.